In [ ]:
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor as RF


# -----------------------------
# Load Dataset
# -----------------------------
df = pd.read_csv("../data/data.csv")


# Select required columns
df = df[
    [
        "source_city",
        "destination_city",
        "airline",
        "class",
        "duration",
        "price"
    ]
]


# Rename columns
df.rename(
    columns={
        "source_city": "from",
        "destination_city": "to"
    },
    inplace=True
)


# -----------------------------
# Filter flights <= 5 hours
# -----------------------------
df = df[df["duration"] <= 5]
# -----------------------------


# ------------------------------------------
# Removing Vistara and Go First 
# Airlines that are nowDiscontinued in India 
# ------------------------------------------
df = df[df["airline"] != "Vistara"] 
df = df[df["airline"] != "'GO_FIRST'"]
# ------------------------------------------


# -----------------------------
# Separate X and y
# -----------------------------
X = df.drop("price", axis=1)
y = df["price"]


# -----------------------------
# One-Hot Encoding
# -----------------------------
encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

encoded = encoder.fit_transform(
    X[["from", "to", "airline", "class"]]
)


# Give encoded columns meaningful names
encoded_df = pd.DataFrame(
    encoded,
    columns=encoder.get_feature_names_out(
        ["from", "to", "airline", "class"]
    ),
    index=X.index
)


# Add duration
X_encoded = pd.concat(
    [
        encoded_df,
        X[["duration"]]
    ],
    axis=1
)


# Make sure all column names are strings
X_encoded.columns = X_encoded.columns.astype(str)


# -----------------------------
# Train-Test Split
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y,
    test_size=0.2,
    random_state=42
)


# -----------------------------
# Random Forest Regressor
# -----------------------------
model = RF(
    random_state=42
)


# Hyperparameter grid
params = {
    "n_estimators": [50, 100, 200],
    "max_depth": [5, 10, 20],
    "min_samples_split": [2, 5, 10]
}


# -----------------------------
# Grid Search
# -----------------------------
grid = GridSearchCV(
    estimator=model,
    param_grid=params,
    cv=5,
    scoring="r2",
    n_jobs=-1
)


# Train
grid.fit(X_train, y_train)


# -----------------------------
# Best Parameters
# -----------------------------
print("Best Parameters:")
print(grid.best_params_)

print("\nBest Cross-Validation R2:")
print(grid.best_score_)





Best Parameters:
{'max_depth': 20, 'min_samples_split': 2, 'n_estimators': 50}

Best Cross-Validation R2:
0.9198218804442366


In [7]:
# -----------------------------
# Test Set Evaluation
# -----------------------------
best_model = grid.best_estimator_

y_pred = best_model.predict(X_test)


r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred) ** 0.5


print("\nTest Set Results:")
print("R2 Score:", round(r2,2))
print("Mean Absolute Error:", round(mae,2))
print("Root Mean Squared Error:", round(rmse,2))


Test Set Results:
R2 Score: 0.92
Mean Absolute Error: 1375.03
Root Mean Squared Error: 2045.21


In [8]:
# -----------------------------
# Train Set Evaluation
# -----------------------------

y_pred = best_model.predict(X_train)


r2 = r2_score(y_train, y_pred)
mae = mean_absolute_error(y_train, y_pred)
rmse = mean_squared_error(y_train, y_pred) ** 0.5


print("\nTrain Set Results:")
print("R2 Score:", round(r2,2))
print("Mean Absolute Error:", round(mae,2))
print("Root Mean Squared Error:", round(rmse,2))


Train Set Results:
R2 Score: 0.93
Mean Absolute Error: 1351.86
Root Mean Squared Error: 2018.1


In [9]:
print(df["from"].unique())
print(df["to"].unique())
print(df["airline"].unique())

<ArrowStringArray>
['Delhi', 'Mumbai', 'Bangalore', 'Kolkata', 'Hyderabad', 'Chennai']
Length: 6, dtype: str
<ArrowStringArray>
['Mumbai', 'Bangalore', 'Kolkata', 'Hyderabad', 'Chennai', 'Delhi']
Length: 6, dtype: str
<ArrowStringArray>
['SpiceJet', 'AirAsia', 'GO_FIRST', 'Indigo', 'Air_India']
Length: 5, dtype: str
